In [ ]:
import requests
import pandas as pd


pd.set_option('display.float_format', '{:.4f}'.format)

In [ ]:
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)",
    "Accept": "application/json",
    # "Accept-Encoding": "gzip, deflate",
    # "Host": "www.sec.gov"
}


def get_cik_dict():
    base_url = "https://www.sec.gov/files/company_tickers.json"
    response = requests.get(base_url, headers=HEADERS)
    
    if response.status_code != 200:
        raise Exception("Failed to fetch ticker data")
    
    return response.json()


def get_cik_from_ticker(ticker, data):
    
    for key, value in data.items():
        if value['ticker'].upper() == ticker.upper():
            return str(value['cik_str']).zfill(10)
    
    return "CIK not found for that ticker."


def get_filings_data(cik):
    filing_url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    filing_data = requests.get(filing_url, headers=HEADERS).json()

    return filing_data["facts"]["us-gaap"]


def get_income_statement(frame, facts, metrics):
    values = []

    for metric, unit in metrics.items():
        try:
            values.append([row["val"] for row in facts[metric]["units"][unit] if "frame" in row and row["frame"] == frame][0])
        except Exception as e:
            raise Exception(f"Failed to get {metric}: {e}")
    
    return pd.DataFrame(data={"metric": metrics.keys(), "value": values})

In [ ]:
CIK_DICT = get_cik_dict()

In [ ]:
# Desired company and earnings report
TICKER = "nvda"
FRAME = "CY2024Q3"

# Get full data
CIK = get_cik_from_ticker(TICKER, CIK_DICT)
FACTS = get_filings_data(CIK)

# Mapping of income statement metrics to units
METRICS = {
    "Revenues": "USD",
    "CostOfRevenue": "USD",
    "GrossProfit": "USD",
    "OperatingExpenses": "USD",
    "OperatingIncomeLoss": "USD",
    "IncomeLossFromContinuingOperationsBeforeIncomeTaxesExtraordinaryItemsNoncontrollingInterest": "USD",
    "IncomeTaxExpenseBenefit": "USD",
    "NetIncomeLoss": "USD",
    "EarningsPerShareBasic": "USD/shares",
    "EarningsPerShareDiluted": "USD/shares",
    "WeightedAverageNumberOfSharesOutstandingBasic": "shares",
    "WeightedAverageNumberOfDilutedSharesOutstanding": "shares",
}

In [ ]:
# Get relevant data
get_income_statement(FRAME, FACTS, METRICS)